# 🩻 ChestNet — Custom CNN from Scratch (Full A→Z Pipeline)

**Dataset assumed:** NIH ChestX-ray14 (`Data_Entry_2017.csv` + `images_001 … images_012` folders) — **multi-label** (one X-ray can have several diseases).
**Framework:** PyTorch · **Input:** grayscale 224×224

| Step | What it does |
|---|---|
| 0 | Install / imports |
| 1 | Config ⚙️ |
| 2 | Helper functions 🧰 |
| 3 | Resize images (run once) 🖼️ |
| 4 | Patient-level train/val/test split (run once) ✂️ |
| 5 | Compute mean/std from train set (run once) 📊 |
| 6 | Dataset + DataLoaders 🗂️ |
| 7 | Custom CNN model 🧠 |
| 8 | Sanity check ✅ |
| 9 | Training 🏋️ |
| 10 | Evaluation on test set 📈 |
| 11 | Single-image prediction + Grad-CAM 🔥 |

> 💡 Run cells **top to bottom**. Steps 3–5 only need to run **once** — after that you can skip them.

## 0️⃣ Install & imports

In [ ]:
import json
import random
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms as T

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             precision_score, recall_score, f1_score,
                             multilabel_confusion_matrix)

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1️⃣ Config ⚙️
Change paths and hyperparameters here only.

In [ ]:
# ---------- Paths ----------
from pathlib import Path

# ---------- Paths ----------
ROOT = Path("/content/drive/MyDrive/ChestNet")
DATA_DIR = ROOT / "data"
RAW_IMAGE_DIR = DATA_DIR / "raw_images"          # put images_001 ... images_012 folders here
IMAGE_DIR = DATA_DIR / "images_256"              # resized copies are written here (step 3)
CSV_PATH = DATA_DIR / "Data_Entry_2017.csv"      # NIH metadata file
SPLIT_DIR = DATA_DIR / "splits"                  # train/val/test CSVs (step 4)
STATS_PATH = DATA_DIR / "norm_stats.json"        # mean/std (step 5)
OUT_DIR = ROOT / "outputs" / "custom_cnn"        # checkpoints, plots, reports
# ---------- Labels (14 NIH diseases) ----------
CLASSES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
NUM_CLASSES = len(CLASSES)

# ---------- Image ----------
RESIZE_TO = 256      # size stored on disk (step 3)
IMG_SIZE = 224       # size fed to the network
IN_CHANNELS = 1      # grayscale X-rays

# ---------- Split ----------
TEST_FRAC = 0.20
VAL_FRAC = 0.10
SEED = 42

# ---------- Training ----------
BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7               # early-stopping patience (epochs)
MAX_POS_WEIGHT = 50.0      # cap for class-imbalance weights

# ⚠️ In Jupyter on Windows, num_workers > 0 usually crashes (classes defined in a
# notebook can't be sent to worker processes). Keep 0 in the notebook.
NUM_WORKERS = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"  # mixed precision only on GPU

OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

## 2️⃣ Helper functions 🧰

In [ ]:
def set_seed(seed: int = SEED):
    """Make results repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True  # faster convs for fixed image size


def load_stats():
    with open(STATS_PATH) as f:
        s = json.load(f)
    return s["mean"], s["std"]


def compute_aucs(labels: np.ndarray, probs: np.ndarray) -> np.ndarray:
    """ROC-AUC per class. NaN if a class has only one label value in this set."""
    aucs = []
    for i in range(labels.shape[1]):
        if len(np.unique(labels[:, i])) < 2:
            aucs.append(np.nan)
        else:
            aucs.append(roc_auc_score(labels[:, i], probs[:, i]))
    return np.array(aucs)


def plot_history(history: dict, save_path=None, show=False):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(epochs, history["train_loss"], label="train")
    ax[0].plot(epochs, history["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(epochs, history["train_auc"], label="train")
    ax[1].plot(epochs, history["val_auc"], label="val")
    ax[1].set_title("Mean ROC-AUC"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show() if show else plt.close()

## 3️⃣ Resize images 🖼️ (run once)
Resizes the raw 1024×1024 PNGs to 256×256 into **one flat folder** → much faster training.

In [ ]:
def resize_one(src):
    dst = IMAGE_DIR / src.name
    if dst.exists():
        return
    img = Image.open(src).convert("L")
    img = img.resize((RESIZE_TO, RESIZE_TO), Image.Resampling.LANCZOS)
    img.save(dst)


def resize_all_images():
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    files = list(RAW_IMAGE_DIR.rglob("*.png"))   # searches images_001/images, images_002/images, ...
    print(f"Found {len(files)} images")
    with ThreadPoolExecutor(max_workers=8) as ex:
        list(tqdm(ex.map(resize_one, files), total=len(files)))
    print("Done ✅ ->", IMAGE_DIR)


resize_all_images()

## 4️⃣ Patient-level split ✂️ (run once)
All images of one patient go into **only one** split → prevents data leakage.

In [ ]:
print("ROOT:", ROOT, "| exists:", ROOT.exists())
print("CSV_PATH:", CSV_PATH, "| exists:", CSV_PATH.exists())
print("RAW_IMAGE_DIR:", RAW_IMAGE_DIR, "| exists:", RAW_IMAGE_DIR.exists())
print("IMAGE_DIR:", IMAGE_DIR, "| exists:", IMAGE_DIR.exists())
print("PNGs in IMAGE_DIR:", len(list(IMAGE_DIR.glob("*.png"))) if IMAGE_DIR.exists() else 0)
print("Contents of DATA_DIR:", [p.name for p in DATA_DIR.iterdir()] if DATA_DIR.exists() else "missing")

In [ ]:
import zipfile, time
t = time.time()
zip_path = DATA_DIR / "The NIH ChestX-ray14 dataset.zip"

with zipfile.ZipFile(zip_path) as z:
    print("Files in zip:", len(z.namelist()))
    print(z.namelist()[:10])          # peek at the structure
    z.extractall("/content/raw_images")

print("Done in", round(time.time() - t), "s")

In [ ]:
RAW_IMAGE_DIR = Path("/content/raw_images")

In [ ]:
!cp -r /content/drive/MyDrive/ChestNet/data/images_256 /content/images_256

In [ ]:
from pathlib import Path

# 1. Did extraction happen?
raw = Path("/content/raw_images")
print("Extracted folder exists:", raw.exists())
if raw.exists():
    pngs = list(raw.rglob("*.png"))
    print("PNGs found after extraction:", len(pngs))
    print("Example:", pngs[:2])

# 2. Is images_256 still empty?
print("PNGs in IMAGE_DIR:", len(list(IMAGE_DIR.glob("*.png"))))

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm import tqdm

RAW_IMAGE_DIR = Path("/content/raw_images")   # extracted images live under here
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

src_files = list(RAW_IMAGE_DIR.rglob("*.png"))
print("Source images:", len(src_files))

for p in tqdm(src_files):
    out = IMAGE_DIR / p.name
    if out.exists():          # skip already-resized files if you re-run
        continue
    img = Image.open(p).convert("L")                 # grayscale, matches IN_CHANNELS = 1
    img = img.resize((RESIZE_TO, RESIZE_TO), Image.BILINEAR)
    img.save(out)

print("PNGs in IMAGE_DIR:", len(list(IMAGE_DIR.glob("*.png"))))

In [ ]:
def prepare_splits():
    df = pd.read_csv(CSV_PATH)[["Image Index", "Finding Labels", "Patient ID"]]

    # Keep only images that exist on disk
    existing = {p.name for p in IMAGE_DIR.glob("*.png")}
    df = df[df["Image Index"].isin(existing)].reset_index(drop=True)
    print(f"Images with metadata on disk: {len(df)}")

    # "Effusion|Mass" -> multi-hot columns. "No Finding" = all zeros.
    split_labels = df["Finding Labels"].str.split("|")
    for c in CLASSES:
        df[c] = split_labels.apply(lambda labels: int(c in labels))

    # ---- Patient-level split ----
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=SEED)
    trainval_idx, test_idx = next(gss.split(df, groups=df["Patient ID"]))
    trainval, test = df.iloc[trainval_idx], df.iloc[test_idx]

    val_size = VAL_FRAC / (1 - TEST_FRAC)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=SEED)
    train_idx, val_idx = next(gss2.split(trainval, groups=trainval["Patient ID"]))
    train, val = trainval.iloc[train_idx], trainval.iloc[val_idx]

    # Leakage check
    tr, va, te = set(train["Patient ID"]), set(val["Patient ID"]), set(test["Patient ID"])
    assert not (tr & va) and not (tr & te) and not (va & te), "Patient overlap found!"

    for name, part in [("train", train), ("val", val), ("test", test)]:
        part.to_csv(SPLIT_DIR / f"{name}.csv", index=False)
        print(f"{name:5s}: {len(part):6d} images | {part['Patient ID'].nunique():5d} patients")

    print("\nPositive count per class (train):")
    print(train[CLASSES].sum().sort_values(ascending=False).to_string())


prepare_splits()